In [1]:
# =======================
# Step 1: Import Libraries
# =======================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

In [2]:
# =======================
# Step 2: Load Dataset
# =======================
df = pd.read_csv("Loan_default_dataset.csv")
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [5]:
# =======================
# Step 3: Preprocessing - Drop ID column
# =======================
df.drop(columns=["LoanID"], inplace=True)

In [7]:
# =======================
# Step 4: Define Features and Target
# =======================
X = df.drop(columns=["Default"])
y = df["Default"]

In [9]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Identify column types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

In [11]:
# =======================
# Step 5: Build Column Transformers with Imputers
# =======================
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


In [13]:
# =======================
# Step 6: Build the Pipeline
# =======================
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])


In [15]:
# Train the full pipeline on training data
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage',
       'HasDependents', 'LoanPurpose', 'HasCoSigner'],
      dtype='object'))])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [17]:
# Predict on test data
y_pred = pipeline.predict(X_test)

# Evaluate model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.8865674564323478
Classification Report:
               precision    recall  f1-score   support

           0       0.89      1.00      0.94     45170
           1       0.69      0.03      0.06      5900

    accuracy                           0.89     51070
   macro avg       0.79      0.52      0.50     51070
weighted avg       0.87      0.89      0.84     51070

Confusion Matrix:
 [[45084    86]
 [ 5707   193]]


In [19]:
# Save the trained pipeline
joblib.dump(pipeline, "loan_default_pipe.pkl")


['loan_default_pipe.pkl']

In [21]:
import pandas as pd
import joblib

test_data = pd.DataFrame([{
    'Age': 35,
    'Income': 60000,
    'LoanAmount': 15000,
    'CreditScore': 680,
    'MonthsEmployed': 24,
    'NumCreditLines': 3,
    'InterestRate': 9.5,
    'LoanTerm': 36,
    'DTIRatio': 0.35,
    'Education': "Bachelor's",
    'EmploymentType': "Full-time",
    'MaritalStatus': "Single",
    'HasMortgage': "No",
    'HasDependents': "No",
    'LoanPurpose': "Personal",
    'HasCoSigner': "No"
}])

model = joblib.load("loan_default_pipe.pkl")
prediction = model.predict(test_data)[0]

print("Prediction:", prediction)


Prediction: 0


In [23]:
!pip install streamlit
